In [16]:
import networkx as nx
import itertools
from math import comb
import matplotlib.pyplot as plt
from typing import List, Tuple, Set

In [3]:
def hamming_distance(a: int, b: int) -> int:
    """Distancia de Hamming entre dos enteros (popcount del XOR)."""
    return (a ^ b).bit_count()

def build_hamming_graph(n: int, d: int) -> nx.Graph:
    """
    Construye el grafo de Hamming H(n, d):
    - Vértices: números enteros de 0 a 2^n - 1 (representan palabras binarias)
    - Arista entre u y v si hamming_distance(u, v) >= d
    """
    num_vertices = 1 << n
    G = nx.Graph()
    G.add_nodes_from(range(num_vertices))
    for u in range(num_vertices):
        for v in range(u + 1, num_vertices):
            if hamming_distance(u, v) >= d:
                G.add_edge(u, v)
    return G


In [9]:
def bron_kerbosch_max_cliques(G: nx.Graph):
    """
    Implementación del algoritmo de Bron–Kerbosch (versión con pivote)
    que encuentra todos los cliques maximales en un grafo no dirigido.
    Adaptado del artículo original "Algorithm 457: Finding All Cliques of an Undirected Graph".
    Retorna una lista de cliques (cada clique es un conjunto de vértices).
    """
    # Convertir los vértices a enteros 0..N-1 (ya lo son)
    vertices = list(G.nodes())
    # Precalcular vecinos como lista de conjuntos para acceso rápido
    neighbors = {v: set(G.neighbors(v)) for v in vertices}
    N = len(vertices)
    # Orden inicial: todos los vértices en "candidates", "not" vacío
    all_vertices = vertices[:]  # lista de todos los vértices en orden
    compsub = []  # R, clique en construcción
    cliques = []  # almacenará los cliques maximales encontrados

    def extend(old, ne, ce):
        """
        old: lista de vértices (primero 'not' (0..ne-1), luego 'candidates' (ne..ce-1))
        ne: número de vértices en not (0 <= ne <= ce)
        ce: número total de elementos en old (len(old))
        """
        nonlocal cliques
        # Paso 1: elegir punto fijo (fixp) con mínimo número de disconexiones
        minnod = ce
        fixp = None
        s = -1
        nod = 0  # indicador si el punto fijo se tomó de candidates (1) o not (0)
        i = 0
        while i < ce and minnod != 0:
            p = old[i]
            count = 0
            # Contar disconexiones con el resto de candidatos (desde ne hasta ce-1)
            j = ne
            pos = -1
            while j < ce and count <= minnod:
                if p not in neighbors[old[j]]:  # disconexión
                    count += 1
                    pos = j
                j += 1
            if count < minnod:
                fixp = p
                minnod = count
                if i < ne:
                    s = pos
                else:
                    s = i
                    nod = 1
            i += 1

        # Bucle principal de backtracking
        # nod iterará desde minnod + nod hasta 1
        for _ in range(minnod + nod, 0, -1):
            # Intercambiar el candidato seleccionado (old[s]) con old[ne]
            p = old[s]
            old[s] = old[ne]
            sel = old[ne]
            old[ne] = p

            # Construir nuevo conjunto 'not' (new) y 'candidates' (newcand)
            new = [0] * ce  # preasignamos tamaño máximo
            newne = 0
            # Copiar vértices de 'not' que son vecinos de sel
            for i in range(ne):
                if sel in neighbors[old[i]]:
                    new[newne] = old[i]
                    newne += 1
            newce = newne
            # Copiar vértices de 'candidates' (desde ne+1 hasta ce-1) que son vecinos de sel
            # Nota: el índice ne ya contiene sel, que fue movido; lo saltamos
            for i in range(ne + 1, ce):
                if sel in neighbors[old[i]]:
                    new[newce] = old[i]
                    newce += 1

            compsub.append(sel)

            if newce == 0:
                # Se encontró un clique maximal
                cliques.append(compsub.copy())
            else:
                if newne < newce:
                    # Llamada recursiva con el nuevo conjunto
                    # new[:newce] contiene (not + candidates)
                    extend(new, newne, newce)

            compsub.pop()
            # Mover sel al conjunto 'not' para futuras iteraciones
            ne += 1

            # Si aún quedan candidatos por procesar (nod > 1), seleccionar el siguiente
            # candidato desconectado del punto fijo
            if nod > 1:
                # Buscar siguiente candidato (pos > s) que esté desconectado de fixp
                s = ne
                while s < ce and fixp in neighbors[old[s]]:
                    s += 1
                if s >= ce:
                    break
                # El siguiente candidato ya está en old[s], listo para el siguiente ciclo
            else:
                # Solo un candidato, terminar
                break

    # Iniciar llamada con todos los vértices en candidates, not vacío
    extend(all_vertices, 0, N)
    return cliques


In [13]:
G= build_hamming_graph(8,4)
cliques = bron_kerbosch_max_cliques(G)
cliques

[[0, 15, 51, 60, 85, 90, 102, 105, 170, 204, 240, 255]]

In [14]:
def main():
    # Parámetros de prueba
    n = 8      # longitud de las palabras
    d = 4      # distancia mínima requerida
    print(f"Construyendo grafo de Hamming H({n},{d})...")
    G = build_hamming_graph(n, d)
    print(f"Vértices: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")

    print("\n=== Cliques maximales encontrados por nuestra implementación ===")
    our_cliques = bron_kerbosch_max_cliques(G)
    print(f"Número de cliques maximales: {len(our_cliques)}")
    max_size = max(len(c) for c in our_cliques) if our_cliques else 0
    print(f"Tamaño del clique máximo (A({n},{d})): {max_size}")
    # Mostrar primeros 5 cliques como ejemplo
    print("Ejemplo de cliques (primeros 5):")
    for i, clique in enumerate(our_cliques[:5]):
        # Convertir enteros a representación binaria para mejor visualización
        bin_repr = [format(v, f'0{n}b') for v in clique]
        print(f"  {i+1}: {bin_repr}")

    # Comparación con networkx.find_cliques (Bron–Kerbosch implementado en C)
    print("\n=== Comparación con networkx.find_cliques ===")
    nx_cliques = list(nx.find_cliques(G))
    print(f"networkx encontró {len(nx_cliques)} cliques maximales.")
    nx_max_size = max(len(c) for c in nx_cliques) if nx_cliques else 0
    print(f"Tamaño del clique máximo según networkx: {nx_max_size}")

    # Verificar que nuestros cliques coinciden (como conjuntos)
    our_sets = [set(c) for c in our_cliques]
    nx_sets = [set(c) for c in nx_cliques]
    if set(frozenset(s) for s in our_sets) == set(frozenset(s) for s in nx_sets):
        print("¡Los conjuntos de cliques maximales coinciden perfectamente!")
    else:
        print("Advertencia: los conjuntos difieren. Revisar implementación.")

if __name__ == "__main__":
    main()

Construyendo grafo de Hamming H(8,4)...
Vértices: 256, Aristas: 20864

=== Cliques maximales encontrados por nuestra implementación ===
Número de cliques maximales: 1
Tamaño del clique máximo (A(8,4)): 12
Ejemplo de cliques (primeros 5):
  1: ['00000000', '00001111', '00110011', '00111100', '01010101', '01011010', '01100110', '01101001', '10101010', '11001100', '11110000', '11111111']

=== Comparación con networkx.find_cliques ===


KeyboardInterrupt: 

In [17]:
def bron_kerbosch_version2(graph: nx.Graph) -> List[Set[int]]:
    """
    Implementación del algoritmo de Bron–Kerbosch versión 2 (con pivote)
    para encontrar todas las cliques maximales.
    Sigue la lógica del artículo "Algorithm 457" (Comm. ACM, 1973).
    """
    # Convertir el grafo a una matriz de adyacencia booleana para acceso rápido
    nodes = list(graph.nodes())
    index_of = {node: i for i, node in enumerate(nodes)}
    n_nodes = len(nodes)
    adj = [[False]*n_nodes for _ in range(n_nodes)]
    for u, v in graph.edges():
        i, j = index_of[u], index_of[v]
        adj[i][j] = adj[j][i] = True
    # Para simplificar, trabajamos con índices enteros 0..n_nodes-1
    # La función recursiva interna usará listas de enteros (índices)
    
    all_cliques = []   # almacenará las cliques encontradas (como conjuntos de nodos originales)
    
    # Algoritmo recursivo: extiende una clique parcial (compsub)
    # old: array de enteros ordenado [not | candidates] (ne = tamaño de not, ce = tamaño total)
    # ne, ce: índices que separan not y candidates en el arreglo old[0:ne] es not, old[ne:ce] es candidates
    def extend(old: List[int], ne: int, ce: int):
        # old tiene longitud ce, los primeros ne son "not", los siguientes ce-ne son "candidates"
        # Se trabaja in-place, pero creamos un nuevo arreglo new para la llamada recursiva.
        
        # Determinar el punto fijo (fixp) y el candidato con mínimo número de desconexiones
        # (versión 2 del artículo)
        minnod = ce
        fixp = -1
        s = -1           # posición del candidato que se usará como pivote
        nod = 0          # bandera: 1 si fixp viene de candidates
        i = 0
        while i < ce and minnod != 0:
            p = old[i]
            # Contar cuántos candidatos (en la parte candidates) NO son adyacentes a p
            cnt = 0
            j = ne
            pos = -1
            while j < ce and cnt <= minnod:
                if not adj[p][old[j]]:
                    cnt += 1
                    pos = j   # posición de ese candidato desconectado
                j += 1
            if cnt < minnod:
                fixp = p
                minnod = cnt
                if i < ne:
                    # fixp viene de not
                    s = pos
                else:
                    # fixp viene de candidates
                    s = i
                    nod = 1
            i += 1
        
        # Ciclo de backtracking: se repite para cada candidato elegido
        # nod = minnod + nod (en el artículo se itera desde minnod+nod hasta 1)
        # Aquí implementamos la lógica de selección del candidato a mover
        for _ in range(minnod + nod, 0, -1):
            # Seleccionar el candidato en posición s, intercambiarlo con old[ne] (el primero de candidates)
            p = old[s]
            # Intercambio
            old[s], old[ne] = old[ne], p
            sel = old[ne]
            # Construir nuevos conjuntos new (candidatos y not) basados en sel
            new = [0] * ce
            newne = 0
            # Primero los que estaban en not y son adyacentes a sel
            for i in range(ne):
                if adj[sel][old[i]]:
                    new[newne] = old[i]
                    newne += 1
            # Luego los que estaban en candidates (old[ne+1:ce]) y son adyacentes a sel
            newce = newne
            for i in range(ne+1, ce):
                if adj[sel][old[i]]:
                    new[newne] = old[i]
                    newne += 1
            # newce es el tamaño de la nueva sección candidates (new[newce:newne])
            # En el algoritmo original: newce = newne (antes de añadir candidates)
            # Pero luego se añaden los candidatos, entonces newne es el total, y newce es el inicio de candidates en new
            # En nuestro new, los primeros newce son los nuevos "not", los siguientes son candidates.
            # Notar que newce es el número de elementos que pasaron de not (adyacentes). Los otros son los nuevos candidates.
            # Ahora se añade sel a compsub
            compsub.append(sel)
            if newne == newce:   # es decir, no hay candidatos nuevos
                # Clique maximal encontrada
                clique_nodes = [nodes[v] for v in compsub]
                all_cliques.append(set(clique_nodes))
            else:
                # Llamada recursiva
                extend(new, newce, newne)
            # Quitar sel de compsub
            compsub.pop()
            # Mover sel a not (incrementar ne)
            ne += 1
            # Si todavía quedan candidatos por procesar (nod > 1 en el artículo)
            if _ > 1:
                # Buscar el siguiente candidato que NO está conectado a fixp
                # (para reducir ramas)
                s = ne
                # Avanzar hasta encontrar un candidato desconectado de fixp
                while s < ce and adj[fixp][old[s]]:
                    s += 1
                if s >= ce:
                    break   # no hay más candidatos que cumplan la condición
        # fin del ciclo for
    
    # Inicialización: ALL contiene todos los nodos (índices)
    all_indices = list(range(n_nodes))
    compsub = []    # pila para la clique actual
    # Llamada inicial: old = all_indices, ne = 0, ce = n_nodes
    extend(all_indices, 0, n_nodes)
    return all_cliques

In [ ]:
# ============== Ejemplo de uso ==============
if __name__ == "__main__":
    n = 8  # longitud de las palabras
    d = 4  # distancia mínima deseada
    print(f"Construyendo grafo de Hamming H({n},{d})...")
    G = build_hamming_graph(n, d)
    print(f"Vértices: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")
    
    print("Buscando todas las cliques maximales con Bron-Kerbosch v2...")
    cliques = bron_kerbosch_version2(G)
    max_size = max(len(c) for c in cliques) if cliques else 0
    print(f"Número de cliques maximales encontradas: {len(cliques)}")
    print(f"Tamaño de la clique máxima: {max_size}")
    print(f"Por lo tanto, A({n},{d}) = {max_size}")
    
    # Mostrar algunas cliques grandes como ejemplo
    print("\nEjemplos de cliques maximales (hasta 5):")
    for i, clique in enumerate(cliques[:5]):
        print(f"  Clique {i+1}: tamaño {len(clique)} -> {clique}")

Construyendo grafo de Hamming H(8,4)...
Vértices: 256, Aristas: 20864
Buscando todas las cliques maximales con Bron-Kerbosch v2...
